# Notebook 02: Data Cleaning & Feature Engineering

## Objective

The goal of this notebook is to clean the raw OULAD datasets, handle missing values, remove unnecessary features, engineer new features, merge all relevant datasets, and prepare a final machine learning dataset for dropout prediction.

By the end of this notebook, we will obtain a clean dataset ready for model training.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

In [2]:
courses = pd.read_csv("../data/courses.csv")
studentInfo = pd.read_csv("../data/studentInfo.csv")
studentRegistration = pd.read_csv("../data/studentRegistration.csv")
studentAssessment = pd.read_csv("../data/studentAssessment.csv")
assessments = pd.read_csv("../data/assessments.csv")
studentVle = pd.read_csv("../data/studentVle.csv")
vle = pd.read_csv("../data/vle.csv")

## Workflow

In this notebook we will perform:

1. Clean individual datasets.
2. Handle missing values and special symbols.
3. Remove unnecessary columns.
4. Create the binary dropout target.
5. Engineer useful behavioral features.
6. Merge all datasets.
7. Prepare the final dataset for machine learning.

In [3]:
dropout_mapping = {
    "Withdrawn": 1,
    "Pass": 0,
    "Fail": 0,
    "Distinction": 0
}
studentInfo["dropout"] = studentInfo["final_result"].map(dropout_mapping)

In [4]:
studentInfo[["final_result", "dropout"]].head(10)

,final_result,dropout
0,Pass,0
1,Pass,0
2,Withdrawn,1
3,Pass,0
4,Pass,0
5,Pass,0
6,Pass,0
7,Pass,0
8,Pass,0
9,Pass,0


In [5]:
studentInfo["dropout"].value_counts()

dropout
0    22437
1    10156
Name: count, dtype: int64

In [6]:
studentInfo.drop(columns=["final_result"], inplace=True)

In [7]:
studentInfo.head()

,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,dropout
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,0
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,0
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,1
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N,0
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,N,0


In [8]:
studentRegistration.head()

,code_module,code_presentation,id_student,date_registration,date_unregistration
0,AAA,2013J,11391,-159,?
1,AAA,2013J,28400,-53,?
2,AAA,2013J,30268,-92,12
3,AAA,2013J,31604,-52,?
4,AAA,2013J,32885,-176,?


In [9]:
studentRegistration.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32593 entries, 0 to 32592
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   code_module          32593 non-null  object
 1   code_presentation    32593 non-null  object
 2   id_student           32593 non-null  int64 
 3   date_registration    32593 non-null  object
 4   date_unregistration  32593 non-null  object
dtypes: int64(1), object(4)
memory usage: 1.2+ MB


In [10]:
studentRegistration.isnull().sum()

code_module            0
code_presentation      0
id_student             0
date_registration      0
date_unregistration    0
dtype: int64

In [11]:
studentRegistration["date_registration"] = studentRegistration["date_registration"].replace("?", np.nan)
studentRegistration["date_unregistration"] = studentRegistration["date_unregistration"].replace("?", np.nan)

In [12]:
studentRegistration["date_registration"] = pd.to_numeric(studentRegistration["date_registration"])
studentRegistration["date_unregistration"] = pd.to_numeric(studentRegistration["date_unregistration"])

In [13]:
studentRegistration.info()
studentRegistration.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32593 entries, 0 to 32592
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   code_module          32593 non-null  object 
 1   code_presentation    32593 non-null  object 
 2   id_student           32593 non-null  int64  
 3   date_registration    32548 non-null  float64
 4   date_unregistration  10072 non-null  float64
dtypes: float64(2), int64(1), object(2)
memory usage: 1.2+ MB


code_module                0
code_presentation          0
id_student                 0
date_registration         45
date_unregistration    22521
dtype: int64

The `?` values in registration dates were converted into `NaN`. In `date_unregistration`, missing values indicate that the student did not unregister, so they should not be filled using mean or median.

In [14]:
studentRegistration = studentRegistration.dropna(subset=["date_registration"])

In [15]:
studentRegistration.isnull().sum()

code_module                0
code_presentation          0
id_student                 0
date_registration          0
date_unregistration    22515
dtype: int64

Cleaning Summary - studentRegistration

- Converted '?' values to NaN.
- Converted registration columns to numeric datatype.
- Removed 45 records with missing registration dates (0.14% of the dataset).
- Retained missing values in `date_unregistration` because they indicate that the student did not unregister from the course.

In [16]:
studentAssessment.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 173912 entries, 0 to 173911
Data columns (total 5 columns):
 #   Column          Non-Null Count   Dtype 
---  ------          --------------   ----- 
 0   id_assessment   173912 non-null  int64 
 1   id_student      173912 non-null  int64 
 2   date_submitted  173912 non-null  int64 
 3   is_banked       173912 non-null  int64 
 4   score           173912 non-null  object
dtypes: int64(4), object(1)
memory usage: 6.6+ MB


In [17]:
studentAssessment["score"].unique()

array(['78', '70', '72', '69', '79', '71', '68', '73', '67', '83', '66',
       '59', '82', '60', '75', '74', '62', '63', '84', '80', '76', '85',
       '57', '81', '87', '77', '45', '65', '61', '52', '54', '51', '88',
       '58', '64', '55', '38', '91', '47', '89', '36', '86', '49', '53',
       '39', '?', '90', '50', '56', '30', '11', '40', '94', '48', '46',
       '25', '34', '42', '18', '37', '28', '33', '95', '35', '44', '41',
       '15', '0', '43', '93', '32', '92', '98', '24', '19', '27', '29',
       '20', '97', '23', '99', '100', '10', '5', '13', '26', '22', '8',
       '12', '16', '9', '96', '14', '21', '17', '31', '6', '1', '7', '4',
       '2', '3'], dtype=object)

In [18]:
studentAssessment["score"].value_counts().head()

score
100    18813
80     12109
60      5620
78      4504
90      4368
Name: count, dtype: int64

In [19]:
studentAssessment[studentAssessment["score"] == "?"].shape

(173, 5)

In [20]:
studentAssessment["score"] = studentAssessment["score"].replace("?", np.nan)

In [21]:
studentAssessment["score"] = pd.to_numeric(studentAssessment["score"])

In [22]:
studentAssessment = studentAssessment.dropna(subset=["score"])

In [23]:
studentAssessment.isnull().sum()

id_assessment     0
id_student        0
date_submitted    0
is_banked         0
score             0
dtype: int64

- Replaced '?' values in the `score` column with `NaN`.
- Converted the `score` column to numeric.
- Removed 173 records with missing scores (approximately 0.1% of the dataset).
- Since the missing percentage was extremely low and the business meaning of missing scores was uncertain, dropping these records was preferred over imputing artificial values.

In [24]:
assessments.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 206 entries, 0 to 205
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   code_module        206 non-null    object 
 1   code_presentation  206 non-null    object 
 2   id_assessment      206 non-null    int64  
 3   assessment_type    206 non-null    object 
 4   date               206 non-null    object 
 5   weight             206 non-null    float64
dtypes: float64(1), int64(1), object(4)
memory usage: 9.8+ KB


In [25]:
assessments.isnull().sum()

code_module          0
code_presentation    0
id_assessment        0
assessment_type      0
date                 0
weight               0
dtype: int64

In [26]:
assessments["date"].value_counts(dropna=False)

date
222    15
229    14
?      11
241     9
227     8
       ..
88      1
123     1
165     1
137     1
149     1
Name: count, Length: 75, dtype: int64

In [27]:
assessments["date"] = assessments["date"].replace("?", np.nan)

In [28]:
assessments["date"] = pd.to_numeric(assessments["date"])

In [29]:
assessments = assessments.dropna(subset=["date"])

In [30]:
assessments.isnull().sum()

code_module          0
code_presentation    0
id_assessment        0
assessment_type      0
date                 0
weight               0
dtype: int64

- Converted '?' values in the `date` column into `NaN`.
- Converted the `date` column to numeric datatype.
- Removed 11 assessment records with missing due dates.
- Since the due date is an important attribute and the missing percentage was small, dropping those records was preferred over imputing artificial values.

In [31]:
studentVle.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10655280 entries, 0 to 10655279
Data columns (total 6 columns):
 #   Column             Dtype 
---  ------             ----- 
 0   code_module        object
 1   code_presentation  object
 2   id_student         int64 
 3   id_site            int64 
 4   date               int64 
 5   sum_click          int64 
dtypes: int64(4), object(2)
memory usage: 487.8+ MB


In [32]:
studentVle.isnull().sum()

code_module          0
code_presentation    0
id_student           0
id_site              0
date                 0
sum_click            0
dtype: int64

- No missing values were found.
- All columns already have appropriate data types.
- No preprocessing was required at this stage.

In [33]:
vle.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6364 entries, 0 to 6363
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   id_site            6364 non-null   int64 
 1   code_module        6364 non-null   object
 2   code_presentation  6364 non-null   object
 3   activity_type      6364 non-null   object
 4   week_from          6364 non-null   object
 5   week_to            6364 non-null   object
dtypes: int64(1), object(5)
memory usage: 298.4+ KB


In [34]:
vle.isnull().sum()

id_site              0
code_module          0
code_presentation    0
activity_type        0
week_from            0
week_to              0
dtype: int64

In [35]:
vle["week_from"].value_counts(dropna=False)

week_from
?     5243
18      91
1       84
28      63
9       52
21      51
27      49
11      47
15      46
29      45
20      43
2       41
10      40
22      38
8       36
3       35
24      33
23      33
5       33
14      32
13      31
26      30
19      29
16      27
12      26
25      24
4       24
6       17
7       15
17       4
0        2
Name: count, dtype: int64

In [36]:
vle["week_to"].value_counts(dropna=False)

week_to
?     5243
18      91
1       80
28      63
9       54
21      51
27      49
11      47
15      46
2       45
29      45
20      43
10      40
22      38
8       34
23      33
24      33
3       33
14      32
5       31
13      31
26      30
19      29
16      27
4       26
12      26
25      24
6       17
7       17
17       4
0        2
Name: count, dtype: int64

In [37]:
vle.drop(columns=["week_from", "week_to"], inplace=True)

- The `week_from` and `week_to` columns contained more than 80% missing values represented by '?'.
- Since these columns provide limited value for dropout prediction and removing the rows would result in a significant loss of data, both columns were removed from the dataset.

In [38]:
courses.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22 entries, 0 to 21
Data columns (total 3 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   code_module                 22 non-null     object
 1   code_presentation           22 non-null     object
 2   module_presentation_length  22 non-null     int64 
dtypes: int64(1), object(2)
memory usage: 660.0+ bytes


In [39]:
courses.isnull().sum()

code_module                   0
code_presentation             0
module_presentation_length    0
dtype: int64

- No missing values were found.
- No preprocessing was required.

In [41]:
base_df = studentInfo.merge(
    studentRegistration,
    on=["code_module", "code_presentation", "id_student"],
    how="left"
)
base_df.shape

(32593, 14)

In [42]:
base_df.head()

,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,dropout,date_registration,date_unregistration
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,0,-159.0,NaN
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,0,-53.0,NaN
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,1,-92.0,12.0
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N,0,-52.0,NaN
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,N,0,-176.0,NaN


In [43]:
base_df.isnull().sum()

code_module                 0
code_presentation           0
id_student                  0
gender                      0
region                      0
highest_education           0
imd_band                    0
age_band                    0
num_of_prev_attempts        0
studied_credits             0
disability                  0
dropout                     0
date_registration          45
date_unregistration     22560
dtype: int64

In [44]:
base_df.dropna(subset=["date_registration"], inplace=True)

In [45]:
base_df.isnull().sum()

code_module                 0
code_presentation           0
id_student                  0
gender                      0
region                      0
highest_education           0
imd_band                    0
age_band                    0
num_of_prev_attempts        0
studied_credits             0
disability                  0
dropout                     0
date_registration           0
date_unregistration     22515
dtype: int64

In [46]:
assessment_df = studentAssessment.merge(
    assessments,
    on="id_assessment",
    how="left"
)
assessment_df.shape

(173739, 10)

In [47]:
assessment_df.head()

,id_assessment,id_student,date_submitted,is_banked,score,code_module,code_presentation,assessment_type,date,weight
0,1752,11391,18,0,78.0,AAA,2013J,TMA,19.0,10.0
1,1752,28400,22,0,70.0,AAA,2013J,TMA,19.0,10.0
2,1752,31604,17,0,72.0,AAA,2013J,TMA,19.0,10.0
3,1752,32885,26,0,69.0,AAA,2013J,TMA,19.0,10.0
4,1752,38053,19,0,79.0,AAA,2013J,TMA,19.0,10.0


In [48]:
assessment_df.isnull().sum()

id_assessment           0
id_student              0
date_submitted          0
is_banked               0
score                   0
code_module          2865
code_presentation    2865
assessment_type      2865
date                 2865
weight               2865
dtype: int64

In [49]:
assessment_df.dropna(subset=["code_module", "code_presentation"], inplace=True)

In [51]:
assessment_df.shape

(170874, 10)

In [52]:
assessment_df.isnull().sum()

id_assessment        0
id_student           0
date_submitted       0
is_banked            0
score                0
code_module          0
code_presentation    0
assessment_type      0
date                 0
weight               0
dtype: int64

In [53]:
assessment_features = assessment_df.groupby(
    ["code_module", "code_presentation", "id_student"]
).agg(
    average_score=("score", "mean"),
    max_score=("score", "max"),
    min_score=("score", "min"),
    total_assessments=("id_assessment", "count"),
    average_weight=("weight", "mean"),
    total_weight=("weight", "sum"),
    average_submission_day=("date_submitted", "mean")
).reset_index()

assessment_features.head()

,code_module,code_presentation,id_student,average_score,max_score,min_score,total_assessments,average_weight,total_weight,average_submission_day
0,AAA,2013J,11391,82.0,85.0,78.0,5,20.0,100.0,112.4
1,AAA,2013J,28400,66.4,70.0,60.0,5,20.0,100.0,114.2
2,AAA,2013J,31604,76.0,88.0,71.0,5,20.0,100.0,112.2
3,AAA,2013J,32885,54.4,75.0,30.0,5,20.0,100.0,125.6
4,AAA,2013J,38053,68.0,79.0,50.0,5,20.0,100.0,116.2


In [54]:
assessment_features.shape


(25819, 10)

In [55]:
assessment_features.isnull().sum()

code_module               0
code_presentation         0
id_student                0
average_score             0
max_score                 0
min_score                 0
total_assessments         0
average_weight            0
total_weight              0
average_submission_day    0
dtype: int64

In [56]:
base_df = base_df.merge(
    assessment_features,
    on=["code_module", "code_presentation", "id_student"],
    how="left"
)
base_df.shape

(32548, 21)

In [57]:
base_df.isnull().sum()

code_module                   0
code_presentation             0
id_student                    0
gender                        0
region                        0
highest_education             0
imd_band                      0
age_band                      0
num_of_prev_attempts          0
studied_credits               0
disability                    0
dropout                       0
date_registration             0
date_unregistration       22515
average_score              6733
max_score                  6733
min_score                  6733
total_assessments          6733
average_weight             6733
total_weight               6733
average_submission_day     6733
dtype: int64

In [58]:
assessment_columns = [
    "average_score",
    "max_score",
    "min_score",
    "total_assessments",
    "average_weight",
    "total_weight",
    "average_submission_day"
]

base_df[assessment_columns] = base_df[assessment_columns].fillna(0)

In [59]:
base_df.isnull().sum()

code_module                   0
code_presentation             0
id_student                    0
gender                        0
region                        0
highest_education             0
imd_band                      0
age_band                      0
num_of_prev_attempts          0
studied_credits               0
disability                    0
dropout                       0
date_registration             0
date_unregistration       22515
average_score                 0
max_score                     0
min_score                     0
total_assessments             0
average_weight                0
total_weight                  0
average_submission_day        0
dtype: int64

In [60]:
vle_features = studentVle.groupby(
    ["code_module", "code_presentation", "id_student"]
).agg(
    total_clicks=("sum_click", "sum"),
    average_clicks=("sum_click", "mean"),
    max_clicks=("sum_click", "max"),
    active_days=("date", "nunique"),
    unique_resources=("id_site", "nunique"),
    first_activity_day=("date", "min"),
    last_activity_day=("date", "max")
).reset_index()
vle_features.head()

,code_module,code_presentation,id_student,total_clicks,average_clicks,max_clicks,active_days,unique_resources,first_activity_day,last_activity_day
0,AAA,2013J,11391,934,4.765306,76,40,55,-5,253
1,AAA,2013J,28400,1435,3.337209,23,80,84,-10,239
2,AAA,2013J,30268,281,3.697368,23,12,22,-10,12
3,AAA,2013J,31604,2158,3.254902,22,123,82,-10,264
4,AAA,2013J,32885,1034,2.937500,22,70,66,-10,247


In [61]:
vle_features.isnull().sum()

code_module           0
code_presentation     0
id_student            0
total_clicks          0
average_clicks        0
max_clicks            0
active_days           0
unique_resources      0
first_activity_day    0
last_activity_day     0
dtype: int64

In [62]:
vle_features.shape

(29228, 10)

In [63]:
base_df = base_df.merge(
    vle_features,
    on=["code_module", "code_presentation", "id_student"],
    how="left"
)
base_df.shape

(32548, 28)

In [64]:
base_df.isnull().sum()

code_module                   0
code_presentation             0
id_student                    0
gender                        0
region                        0
highest_education             0
imd_band                      0
age_band                      0
num_of_prev_attempts          0
studied_credits               0
disability                    0
dropout                       0
date_registration             0
date_unregistration       22515
average_score                 0
max_score                     0
min_score                     0
total_assessments             0
average_weight                0
total_weight                  0
average_submission_day        0
total_clicks               3327
average_clicks             3327
max_clicks                 3327
active_days                3327
unique_resources           3327
first_activity_day         3327
last_activity_day          3327
dtype: int64

In [65]:
vle_columns = [
    "total_clicks",
    "average_clicks",
    "max_clicks",
    "active_days",
    "unique_resources",
    "first_activity_day",
    "last_activity_day"
]
base_df[vle_columns] = base_df[vle_columns].fillna(0)

In [66]:
base_df.isnull().sum()

code_module                   0
code_presentation             0
id_student                    0
gender                        0
region                        0
highest_education             0
imd_band                      0
age_band                      0
num_of_prev_attempts          0
studied_credits               0
disability                    0
dropout                       0
date_registration             0
date_unregistration       22515
average_score                 0
max_score                     0
min_score                     0
total_assessments             0
average_weight                0
total_weight                  0
average_submission_day        0
total_clicks                  0
average_clicks                0
max_clicks                    0
active_days                   0
unique_resources              0
first_activity_day            0
last_activity_day             0
dtype: int64

In [67]:
student_vle_activity = studentVle.merge(
    vle,
    on=["id_site", "code_module", "code_presentation"],
    how="left"
)
student_vle_activity.shape

(10655280, 7)

In [68]:
student_vle_activity.head()

,code_module,code_presentation,id_student,id_site,date,sum_click,activity_type
0,AAA,2013J,28400,546652,-10,4,forumng
1,AAA,2013J,28400,546652,-10,1,forumng
2,AAA,2013J,28400,546652,-10,1,forumng
3,AAA,2013J,28400,546614,-10,11,homepage
4,AAA,2013J,28400,546714,-10,1,oucontent


In [69]:
student_vle_activity.isnull().sum()

code_module          0
code_presentation    0
id_student           0
id_site              0
date                 0
sum_click            0
activity_type        0
dtype: int64

In [70]:
activity_features = student_vle_activity.pivot_table(
    index=["code_module", "code_presentation", "id_student"],
    columns="activity_type",
    values="sum_click",
    aggfunc="sum",
    fill_value=0
).reset_index()
activity_features.head()

activity_type,code_module,code_presentation,id_student,dataplus,dualpane,externalquiz,folder,forumng,glossary,homepage,...,ouelluminate,ouwiki,page,questionnaire,quiz,repeatactivity,resource,sharedsubpage,subpage,url
0,AAA,2013J,11391,0,0,0,0,193,0,138,...,0,0,0,0,0,0,13,0,32,5
1,AAA,2013J,28400,10,0,0,0,417,0,324,...,0,0,0,0,0,0,12,0,87,48
2,AAA,2013J,30268,0,0,0,0,126,0,59,...,0,0,0,0,0,0,4,0,22,4
3,AAA,2013J,31604,2,0,0,0,634,1,432,...,0,0,0,0,0,0,19,0,144,90
4,AAA,2013J,32885,0,0,0,0,194,4,204,...,0,0,0,0,0,0,45,0,79,14


In [71]:
activity_features.shape

(29228, 23)

In [72]:
activity_features.columns

Index(['code_module', 'code_presentation', 'id_student', 'dataplus',
       'dualpane', 'externalquiz', 'folder', 'forumng', 'glossary', 'homepage',
       'htmlactivity', 'oucollaborate', 'oucontent', 'ouelluminate', 'ouwiki',
       'page', 'questionnaire', 'quiz', 'repeatactivity', 'resource',
       'sharedsubpage', 'subpage', 'url'],
      dtype='object', name='activity_type')

In [73]:
base_df = base_df.merge(
    activity_features,
    on=["code_module", "code_presentation", "id_student"],
    how="left"
)
base_df.shape

(32548, 48)

In [74]:
base_df.isnull().sum()

code_module                   0
code_presentation             0
id_student                    0
gender                        0
region                        0
highest_education             0
imd_band                      0
age_band                      0
num_of_prev_attempts          0
studied_credits               0
disability                    0
dropout                       0
date_registration             0
date_unregistration       22515
average_score                 0
max_score                     0
min_score                     0
total_assessments             0
average_weight                0
total_weight                  0
average_submission_day        0
total_clicks                  0
average_clicks                0
max_clicks                    0
active_days                   0
unique_resources              0
first_activity_day            0
last_activity_day             0
dataplus                   3327
dualpane                   3327
externalquiz               3327
folder  

In [75]:
activity_columns = activity_features.columns.difference(
    ["code_module", "code_presentation", "id_student"]
)
base_df[activity_columns] = base_df[activity_columns].fillna(0)

In [76]:
base_df.isnull().sum()

code_module                   0
code_presentation             0
id_student                    0
gender                        0
region                        0
highest_education             0
imd_band                      0
age_band                      0
num_of_prev_attempts          0
studied_credits               0
disability                    0
dropout                       0
date_registration             0
date_unregistration       22515
average_score                 0
max_score                     0
min_score                     0
total_assessments             0
average_weight                0
total_weight                  0
average_submission_day        0
total_clicks                  0
average_clicks                0
max_clicks                    0
active_days                   0
unique_resources              0
first_activity_day            0
last_activity_day             0
dataplus                      0
dualpane                      0
externalquiz                  0
folder  

In [77]:
base_df.drop(columns=["date_unregistration"], inplace=True)

as it may create data lekage..

In [78]:
base_df.drop(columns=["id_student"], inplace=True)

In [79]:
base_df.shape

(32548, 46)

In [80]:
base_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32548 entries, 0 to 32547
Data columns (total 46 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   code_module             32548 non-null  object 
 1   code_presentation       32548 non-null  object 
 2   gender                  32548 non-null  object 
 3   region                  32548 non-null  object 
 4   highest_education       32548 non-null  object 
 5   imd_band                32548 non-null  object 
 6   age_band                32548 non-null  object 
 7   num_of_prev_attempts    32548 non-null  int64  
 8   studied_credits         32548 non-null  int64  
 9   disability              32548 non-null  object 
 10  dropout                 32548 non-null  int64  
 11  date_registration       32548 non-null  float64
 12  average_score           32548 non-null  float64
 13  max_score               32548 non-null  float64
 14  min_score               32548 non-null

In [81]:
base_df.head()

,code_module,code_presentation,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,...,ouelluminate,ouwiki,page,questionnaire,quiz,repeatactivity,resource,sharedsubpage,subpage,url
0,AAA,2013J,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,...,0.0,0.0,0.0,0.0,0.0,0.0,13.0,0.0,32.0,5.0
1,AAA,2013J,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,...,0.0,0.0,0.0,0.0,0.0,0.0,12.0,0.0,87.0,48.0
2,AAA,2013J,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,...,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,22.0,4.0
3,AAA,2013J,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N,...,0.0,0.0,0.0,0.0,0.0,0.0,19.0,0.0,144.0,90.0
4,AAA,2013J,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,N,...,0.0,0.0,0.0,0.0,0.0,0.0,45.0,0.0,79.0,14.0


In [82]:
base_df.to_csv("../data/processed/final_dataset.csv", index=False)

In [83]:
base_df.columns.tolist()

['code_module',
 'code_presentation',
 'gender',
 'region',
 'highest_education',
 'imd_band',
 'age_band',
 'num_of_prev_attempts',
 'studied_credits',
 'disability',
 'dropout',
 'date_registration',
 'average_score',
 'max_score',
 'min_score',
 'total_assessments',
 'average_weight',
 'total_weight',
 'average_submission_day',
 'total_clicks',
 'average_clicks',
 'max_clicks',
 'active_days',
 'unique_resources',
 'first_activity_day',
 'last_activity_day',
 'dataplus',
 'dualpane',
 'externalquiz',
 'folder',
 'forumng',
 'glossary',
 'homepage',
 'htmlactivity',
 'oucollaborate',
 'oucontent',
 'ouelluminate',
 'ouwiki',
 'page',
 'questionnaire',
 'quiz',
 'repeatactivity',
 'resource',
 'sharedsubpage',
 'subpage',
 'url']

In [84]:
base_df.rename(columns={
    "forumng": "forum_clicks",
    "homepage": "homepage_clicks",
    "oucontent": "content_clicks",
    "resource": "resource_clicks",
    "quiz": "quiz_clicks",
    "url": "url_clicks",
    "page": "page_clicks",
    "folder": "folder_clicks",
    "glossary": "glossary_clicks",
    "dataplus": "dataplus_clicks",
    "dualpane": "dualpane_clicks",
    "externalquiz": "external_quiz_clicks",
    "htmlactivity": "html_activity_clicks",
    "oucollaborate": "collaborate_clicks",
    "ouelluminate": "elluminate_clicks",
    "ouwiki": "wiki_clicks",
    "questionnaire": "questionnaire_clicks",
    "repeatactivity": "repeat_activity_clicks",
    "sharedsubpage": "shared_subpage_clicks",
    "subpage": "subpage_clicks"
}, inplace=True)

In [85]:
base_df.to_csv("../data/processed/final_dataset.csv",index=False)

# Final Feature Categories

### Demographic Features
- gender
- region
- highest_education
- imd_band
- age_band
- studied_credits
- disability
- num_of_prev_attempts

### Registration Feature
- date_registration

### Assessment Features
- average_score
- max_score
- min_score
- total_assessments
- average_weight
- total_weight
- average_submission_day

### Engagement Features
- total_clicks
- average_clicks
- max_clicks
- active_days
- unique_resources
- first_activity_day
- last_activity_day

### Activity-wise Engagement Features
- forum_clicks
- homepage_clicks
- content_clicks
- resource_clicks
- quiz_clicks
- url_clicks
- page_clicks
- glossary_clicks
- ...